In [38]:
import ARLreader as Ar
import numpy as np
import pandas as pd

def calculate_P(b):                                    #set function to calculate erosion potential
    if b <= 0.172:
        return 0
    else:
        return 58 * (b - 0.172)**2 + 25 * (b - 0.172)

####### Input month and week ###################
month = 'jun24'
week = 'w1'
################################################

potosi_lat = -19.587425                             #set parameters
potosi_lon = -65.771967
K = 0.41
d = 0.0014
k_PM10 = 0.5
N = 1

print(f"Processing {month}.{week}...\n")

filepath = rf'C:\HYSPLIT\working\gdas1.{month}.{week}'
gdas = Ar.reader(filepath)

results = []

# Determine day range based on week
if week == 'w1':
    day_range = range(8)
elif week == 'w2':
    day_range = range(8, 15)
elif week == 'w3':
    day_range = range(15, 22)
else:
    day_range = range(22, 29)

for day_idx in day_range:
    for hour in [0, 3, 6, 9, 12, 15, 18, 21]:
        try:
            recinfo_u, grid, u_wind = gdas.load_heightlevel(day_idx, hour, 0, 'U10M')
            recinfo_v, grid, v_wind = gdas.load_heightlevel(day_idx, hour, 0, 'V10M')
            
            # Find nearest grid point to Potosí (use actual grid shape, not headerinfo)
            Ny, Nx = u_wind.shape
            lats = np.linspace(90, -90, Ny)
            lons = np.linspace(0, 359, Nx)
            lat_idx = np.argmin(np.abs(lats - potosi_lat))
            lon_idx = np.argmin(np.abs(lons - potosi_lon))
            
            u = u_wind[lat_idx, lon_idx]
            v = v_wind[lat_idx, lon_idx]
            
            if np.isnan(u) or np.isnan(v):
                continue

            #calculate friction velocity (b), erosion potential (P), and emission factor (E)
            wind_speed = np.sqrt(u**2 + v**2)
            b = (K * wind_speed) / np.log(10 / d)
            P = calculate_P(b)
            E_PM10 = k_PM10 * N * P

            #create date string
            date_str = f"{recinfo_u.y}-{recinfo_u.m}-{recinfo_u.d}"

            #create table of results
            results.append({
                'date': date_str,
                'hour': hour,
                'wind_speed_ms': round(wind_speed, 4),
                'E_PM10': round(E_PM10, 4)
            })
        except:
            pass

print(f"Results: {len(results)}\n")

#calculate the daily mean of wind speed and emission rate
if len(results) > 0:
    df = pd.DataFrame(results)
    daily = df.groupby('date').agg({'wind_speed_ms': 'mean', 'E_PM10': 'mean'}).round(4)
    print(daily)
    daily.to_csv(f'potosi_{month}_{week}_results.csv')
    print(f"\nSaved to potosi_{month}_{week}_results.csv")

Processing jun24.w1...

indexinfo 
raw header GDAS 0 0 90.0 359.0 1.0 1.0 0.0 0.0 0.0 1.0 1.0 -90.0 0.0 0.0 360 181 24 2 1604
headerinfo source fcth minDatatime griddef Nx Ny Nz Coordzflag headerlength
{0: {'level': 0.0, 'vars': [('PRSS', 85), ('MSLP', 187), ('TPP6', 0), ('UMOF', 0), ('VMOF', 0), ('SHTF', 0), ('DSWF', 0), ('RH2M', 188), ('U10M', 120), ('V10M', 17), ('T02M', 93), ('TCLD', 60), ('SHGT', 49), ('CAPE', 135), ('CINH', 87), ('LISD', 185), ('LIB4', 214), ('PBLH', 28), ('TMPS', 62), ('CPPA', 0), ('SOLM', 46), ('CSNO', 60), ('CICE', 60), ('CFZR', 60), ('CRAI', 186), ('LHTF', 0), ('LCLD', 60), ('MCLD', 60), ('HCLD', 60)]}, 1: {'level': 1000.0, 'vars': [('HGTS', 219), ('TEMP', 127), ('UWND', 84), ('VWND', 253), ('WWND', 25), ('RELH', 162)]}, 2: {'level': 975.0, 'vars': [('HGTS', 220), ('TEMP', 137), ('UWND', 112), ('VWND', 9), ('WWND', 11), ('RELH', 116)]}, 3: {'level': 950.0, 'vars': [('HGTS', 225), ('TEMP', 92), ('UWND', 214), ('VWND', 249), ('WWND', 17), ('RELH', 215)]}, 4: {'

In [39]:
#combine tables
import glob

# Load all CSV files
csv_files = glob.glob('potosi_*_results.csv')
dfs = [pd.read_csv(f) for f in sorted(csv_files)]

# Combine into one dataframe
combined_df = pd.concat(dfs, ignore_index=True)

# Sort by date
combined_df = combined_df.sort_values('date').reset_index(drop=True)

print(combined_df)
print(f"\nTotal rows: {len(combined_df)}")

# Save combined
combined_df.to_csv('potosi_all_results.csv', index=False)
print("\nSaved to potosi_all_results.csv")

       date  wind_speed_ms  E_PM10
0    24-5-1         5.1787  1.0050
1    24-5-2         6.2956  1.9394
2    24-5-3         5.8513  1.6551
3    24-5-4         4.3201  0.4714
4    24-5-5         6.0340  1.9400
5    24-5-6         5.4848  1.3635
6    24-5-7         7.0542  2.6889
7    24-6-1         4.8744  0.7553
8    24-6-2         4.4344  0.4551
9    24-6-3         2.8244  0.0432
10   24-6-4         4.7078  0.7321
11   24-6-5         4.2891  0.4707
12   24-6-6         4.3188  0.3702
13   24-6-7         4.2170  0.4219
14   24-7-1         4.1152  0.2930
15  24-7-10         4.4678  0.4723
16  24-7-11         5.3186  1.0817
17  24-7-12         4.0333  0.2251
18  24-7-13         4.3083  0.3613
19  24-7-14         4.2238  0.3096
20  24-7-15         4.1611  0.2677
21  24-7-16         2.6698  0.0000
22  24-7-17         4.2139  0.4499
23  24-7-18         4.3986  0.5013
24  24-7-19         4.3500  0.4180
25   24-7-2         3.3950  0.2420
26  24-7-20         5.2895  1.1062
27  24-7-21         

In [24]:
mean_EPM10 = combined_df['E_PM10'].mean()

In [25]:
mean_EPM10

np.float64(2.2874375)

In [26]:
mean_monthly = combined_df['E_PM10'].mean()

In [27]:
#find the three days closest to the mean wind speed
combined_df['distance'] = (combined_df['E_PM10'] - mean_EPM10).abs()
three_closest = combined_df.nsmallest(3, 'distance')
combined_df.drop('distance', axis=1, inplace=True)  # Clean up helper column
three_closest

,date,wind_speed_ms,E_PM10,distance
14,24-7-22,5.2647,2.2952,0.007763
26,24-7-8,5.3831,2.2958,0.008363
39,24-9-2,5.3740,2.2781,0.009337
